In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine

host     = '127.0.0.1'
port     = 3306
username = 'root'
password = 'asusrog468'
database = 'BoatSalesDB'

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

os.makedirs(r"C:\Users\Hello\Downloads\project\BoatSalesProject\excel", exist_ok=True)
print("✅ Connected!")

In [ ]:
# montly revenue trend
q1 = """
SELECT
    YEAR(OrderDate)            AS Year,
    MONTH(OrderDate)           AS Month,
    COUNT(SalesID)             AS TotalOrders,
    ROUND(SUM(NetSales), 2)    AS TotalRevenue,
    ROUND(SUM(Profit), 2)      AS TotalProfit,
    ROUND(AVG(DiscountPct), 2) AS AvgDiscount
FROM salesdata
GROUP BY YEAR(OrderDate), MONTH(OrderDate)
ORDER BY YEAR(OrderDate), MONTH(OrderDate)
"""
q1_result = pd.read_sql(q1, engine)
print("✅ Query 1 done")
print(q1_result.head())

In [ ]:
#channel performence
q2 = """
SELECT
    c.ChannelName,
    c.ChannelType,
    COUNT(s.SalesID)          AS TotalOrders,
    ROUND(SUM(s.NetSales), 2) AS TotalRevenue,
    ROUND(SUM(s.Profit),  2)  AS TotalProfit,
    ROUND(SUM(s.Profit)*100.0 / SUM(s.NetSales), 2) AS ProfitMargin_Pct
FROM salesdata s
JOIN saleschannel c ON s.ChannelID = c.ChannelID
GROUP BY c.ChannelName, c.ChannelType
ORDER BY TotalRevenue DESC
"""
q2_result = pd.read_sql(q2, engine)
print("✅ Query 2 done")
print(q2_result)

In [ ]:
#top 10 product
q3 = """
SELECT
    p.ProductName,
    p.Brand,
    p.Category,
    COUNT(s.SalesID)             AS OrderCount,
    ROUND(SUM(s.Profit), 2)      AS TotalProfit,
    ROUND(AVG(s.DiscountPct), 2) AS AvgDiscount
FROM salesdata s
JOIN productdetails p ON s.ProductID = p.ProductID
GROUP BY p.ProductName, p.Brand, p.Category
ORDER BY TotalProfit DESC
limit 10
"""
q3_result = pd.read_sql(q3, engine)
print("✅ Query 3 done")
print(q3_result)

In [ ]:
#customer tier analysis
q4 = """
SELECT
    CustomerTier,
    CustomerSegment,
    COUNT(SalesID)          AS TotalOrders,
    ROUND(SUM(NetSales), 2) AS TotalRevenue,
    ROUND(AVG(NetSales), 2) AS AvgOrderValue,
    ROUND(SUM(Profit),  2)  AS TotalProfit
FROM salesdata
GROUP BY CustomerTier, CustomerSegment
ORDER BY TotalRevenue DESC
"""
q4_result = pd.read_sql(q4, engine)
print("✅ Query 4 done")
print(q4_result)

In [ ]:
#festival impact
q5 = """
SELECT
    IsFestivalPeriod,
    FestivalName,
    COUNT(SalesID)             AS Orders,
    ROUND(SUM(NetSales), 2)    AS Revenue,
    ROUND(AVG(NetSales), 2)    AS AvgOrderValue,
    ROUND(AVG(DiscountPct), 2) AS AvgDiscount
FROM salesdata
GROUP BY IsFestivalPeriod, FestivalName
ORDER BY Revenue DESC
"""
q5_result = pd.read_sql(q5, engine)
print("✅ Query 5 done")
print(q5_result)

In [ ]:
#cancellatio & returns
q6 = """
SELECT
    OrderStatus,
    CancellationReason,
    ReturnReason,
    COUNT(SalesID) AS Count,
    ROUND(COUNT(SalesID)*100.0 /
          (SELECT COUNT(*) FROM salesdata), 2) AS Pct_of_Total
FROM salesdata
WHERE OrderStatus IN ('Cancelled', 'Returned')
GROUP BY OrderStatus, CancellationReason, ReturnReason
ORDER BY Count DESC
"""
q6_result = pd.read_sql(q6, engine)
print("✅ Query 6 done")
print(q6_result)

In [ ]:
#delivery patner performance
q7 = """
SELECT
    d.PartnerName,
    d.Region,
    d.VehicleType,
    COUNT(s.SalesID)              AS TotalDeliveries,
    ROUND(AVG(s.DeliveryDays), 2) AS AvgDeliveryDays,
    ROUND(AVG(d.Rating), 2)       AS AvgRating,
    SUM(CASE WHEN s.OrderStatus = 'Delivered' THEN 1 ELSE 0 END) AS SuccessCount
FROM salesdata s
JOIN deliverypartner d ON s.DeliveryPartnerID = d.DeliveryPartnerID
GROUP BY d.PartnerName, d.Region, d.VehicleType
ORDER BY AvgDeliveryDays ASC
"""
q7_result = pd.read_sql(q7, engine)
print("✅ Query 7 done")
print(q7_result)

In [ ]:
#rfm table
q8 = """
SELECT
    CustomerID,
    DATEDIFF(now(), MAX(OrderDate)) AS Recency_Days,
    COUNT(SalesID)          AS Frequency,
    ROUND(SUM(NetSales), 2) AS Monetary
FROM salesdata
WHERE OrderStatus = 'Delivered'
GROUP BY CustomerID
ORDER BY Monetary DESC
"""
q8_result = pd.read_sql(q8, engine)
print("✅ Query 8 done")
print(q8_result.head(10))

In [ ]:
q9 = """
SELECT
    s.CityTier,
    cu.State,
    COUNT(s.SalesID)          AS Orders,
    ROUND(SUM(s.NetSales), 2) AS Revenue,
    ROUND(AVG(s.NetSales), 2) AS AvgOrderValue
FROM salesdata s
JOIN customerdetails cu ON s.CustomerID = cu.CustomerID
GROUP BY s.CityTier, cu.State
ORDER BY Revenue DESC
"""
q9_result = pd.read_sql(q9, engine)
print("✅ Query 9 done")
print(q9_result.head(10))

In [ ]:
#payment method
q10 = """
SELECT
    CustomerTier,
    PaymentMethod,
    COUNT(SalesID)          AS OrderCount,
    ROUND(SUM(NetSales), 2) AS Revenue
FROM salesdata
GROUP BY CustomerTier, PaymentMethod
ORDER BY CustomerTier, Revenue DESC
"""
q10_result = pd.read_sql(q10, engine)
print("✅ Query 10 done")
print(q10_result)

In [ ]:
excel_path = r"C:\Users\Hello\Downloads\project\BoatSalesProject\excel\sql_query_results.xlsx"

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    q1_result.to_excel(writer,  sheet_name='Monthly_Trend',     index=False)
    q2_result.to_excel(writer,  sheet_name='Channel_Perf',      index=False)
    q3_result.to_excel(writer,  sheet_name='Top10_Products',    index=False)
    q4_result.to_excel(writer,  sheet_name='Customer_Tier',     index=False)
    q5_result.to_excel(writer,  sheet_name='Festival_Impact',   index=False)
    q6_result.to_excel(writer,  sheet_name='Cancellations',     index=False)
    q7_result.to_excel(writer,  sheet_name='Delivery_Partners', index=False)
    q8_result.to_excel(writer,  sheet_name='RFM_Table',         index=False)
    q9_result.to_excel(writer,  sheet_name='Geographic',        index=False)
    q10_result.to_excel(writer, sheet_name='Payment_Methods',   index=False)

print("✅ All 10 queries exported!")
print(f"Saved at: {excel_path}")